# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://mlcommons.github.io/data/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Croissant schema:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}\n{dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets by @id and name
record_sets = dataset.metadata.record_sets
for rec in record_sets:
    print(f"Record set: {rec['@id']} | name: {rec.get('name','')}\n  description: {rec.get('description','')}")
    fields = rec.get('fields', [])
    print("  Fields:")
    for field in fields:
        field_id = field.get('@id')
        label = field.get('name', '')
        description = field.get('description', '')
        dtype = field.get('dataType','')
        print(f"    - {field_id} | {label} | type: {dtype}")
    print("-")

## 3. Data Extraction
Load one or more record sets into pandas DataFrames for further exploration. Use the record set and field `@id`s from above.

In [ ]:
# Example: extract all record sets into a dict
import warnings
warnings.filterwarnings('ignore')

# List all record set IDs
record_set_ids = [rec['@id'] for rec in dataset.metadata.record_sets]

dataframes = {}
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded record set {rec_id} with {df.shape[0]} rows and {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not load record set {rec_id}: {e}")
        continue

# Select key record set for demonstration (primary table):
main_record_set_id = None
if record_set_ids:
    main_record_set_id = record_set_ids[0]

if main_record_set_id is not None:
    print(f"\nColumns of main table ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates outlier removal, value normalization, and grouping.

**All fields are referenced by their `@id`.**

In [ ]:
# Choose a numeric field @id for demonstration (e.g., 'coverage_percentage')
# You can inspect all field IDs above. Here we make a best guess based on expected proteomics fields.

numeric_field_id = None
# Try to intelligently select a numeric field
for field in dataset.metadata.record_sets[0].get('fields', []):
    if field.get('dataType', '').lower() in ['float', 'integer', 'number']:
        numeric_field_id = field['@id']
        break

df = dataframes[main_record_set_id]

if numeric_field_id and numeric_field_id in df.columns:
    print(f"Numeric field chosen: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    # Filtering
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std()!=0 else 1)
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try to group by any non-numeric field
    group_field_id = None
    for field in dataset.metadata.record_sets[0].get('fields', []):
        if field.get('dataType', '').lower() not in ['float', 'integer', 'number']:
            candidate = field['@id']
            if candidate in df.columns:
                group_field_id = candidate
                break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If a group field, show boxplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated loading, overviewing, and exploratory analysis of the FAIR<sup>2</sup> protein mass spectrometry dataset using the `mlcroissant` library. Using the automatic schema inspection by `@id`, you can extend these templates to perform custom analyses and visualizations for your own research.